![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Turning one flat file into a relational database: clean the data, split it into tables, wire up the keys, load it into SQLite.

> **Done when:** `data/project.db` exists with 3+ related tables, `sql/schema.sql` matches your ERD, no foreign key is dangling, and the row counts are what you expect.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

---
## 1. Load the raw data

In [2]:
data = pd.read_csv('../data/raw/Ecommerce_Recommendation_Dataset.csv')

df = data.copy()

---
## 2. Clean

Work through the problems you listed in notebook 01. Every decision you make here — drop the nulls or keep them, cast or discard — is one you may be asked to defend on Friday, so note why.

As mentioned in 01_eda.ipynb Notebook, the following columns are unnecessary for the purpose of this analysis and will therefore be dropped. They are unnecessary because they are outputs of an algorithm.

In [3]:
# Calling clean_data function from functions.py

drop_columns = [
    'Context_Relevance_Score',
    'User_Item_Affinity_Score',	
    'User_Node_Degree',	
    'Item_Node_Degree',	
    'Common_Neighbor_Count',	
    'Attention_Weight',	
    'Embedding_Similarity_Score',	
    'Candidate_Rank',	
    'Distance_Score',	
    'Retrieval_Latency_ms',	
    'Candidate_Recall_Score',	
    'Current_Ranking_Position',	
    'Q_Value_Score',	
    'Reward_Score',	
    'Ranking_Stability_Score',	
    'Recommendation_Action',
    'Final_Recommendation_Score' 
]

df_clean = clean_data(drop_columns, df)

Columns before: 47
Columns after: 30


Stripping whitespaces for categorical columns.

In [4]:
cat = df_clean.select_dtypes(exclude="number")
for c in cat:
    df_clean[c]= df_clean[c].str.strip()

Double check nulls and duplicates after applying the changes above.

In [5]:
nulls = df_clean.isna().sum().sum()
print(f"Total Nulls: {nulls}")

dupes = df_clean.duplicated().sum()
print(f"Total Duplicates: {dupes}")

Total Nulls: 0
Total Duplicates: 0


Overall this dataset is clean and structured, as it's a synthetic dataset. Therefore there aren't many steps to be taken for imputing nulls, cleaning duplicates, etc.

---
## 3. Design your tables

Decide what your tables are and how they relate before you write any code. Draw the ERD (Excalidraw, draw.io), export it as an image, commit it, and link it from your README.

Three tables minimum, with real primary and foreign keys.

---
## 4. Build the tables

Split the cleaned data into the tables you designed.

First step, create all the lookup tables with their own primary keys.

In [6]:
lookup_tbls = {
    'age_groups':        ('Age_Group', 'age_group_id'),
    'genders':            ('Gender', 'gender_id'),
    'membership_levels':  ('Membership_Level', 'membership_id'),
    'location_zones':     ('Location_Zone', 'location_id'),
    'item_categories':    ('Item_Category', 'category_id'),
    'brand_tiers':        ('Brand_Tier', 'brand_tier_id'),
    'device_types':       ('Device_Type', 'device_id'),
    'traffic_sources':    ('Traffic_Source', 'traffic_id'),
}

for tbl_name, (col, col_id) in lookup_tbls.items():
    l_tbl = make_lookup(df_clean, col, col_id)
    df_clean = df_clean.merge(l_tbl, on=col, how='left').drop(columns=[col])
    l_tbl.columns = [c.lower() for c in l_tbl.columns]
    l_tbl.to_csv(f'../data/clean/{tbl_name}.csv', index=False)

Second step, create the interactions table (main).

In [7]:
# Imputing a primary key for the interactions table with an interaction_id.
df_clean.insert(0, 'interaction_id', range(1, len(df_clean) + 1))

# Bring all column names to lowercase to match the lookup tables 
df_clean.columns = [col.lower() for col in df_clean.columns]
print(df_clean.columns.tolist())    # sanity check

# Export to csv
df_clean.to_csv('../data/clean/interactions.csv', index=False)

['interaction_id', 'user_id', 'item_id', 'session_id', 'user_activity_score', 'user_loyalty_score', 'average_order_value', 'past_purchase_count', 'item_price', 'discount_percentage', 'item_popularity_score', 'item_quality_score', 'stock_availability_score', 'click_count', 'view_duration_sec', 'cart_add_count', 'wishlist_flag', 'purchase_flag', 'rating', 'return_flag', 'engagement_score', 'time_of_day', 'day_type', 'age_group_id', 'gender_id', 'membership_id', 'location_id', 'category_id', 'brand_tier_id', 'device_id', 'traffic_id']


---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

In [8]:
for _, col_id in lookup_tbls.values():
    missing = df_clean[col_id].isna().sum()
    print(f"{col_id}: {missing} unmatched")

age_group_id: 0 unmatched
gender_id: 0 unmatched
membership_id: 0 unmatched
location_id: 0 unmatched
category_id: 0 unmatched
brand_tier_id: 0 unmatched
device_id: 0 unmatched
traffic_id: 0 unmatched


Foreign keys validated. 

---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

Connect to SQLite and create the empty tables in schema.sql.

In [10]:
import sqlite3

conn = sqlite3.connect('../data/project.db')
conn.execute("PRAGMA foreign_keys = ON;")

with open('../sql/schema.sql') as o:
    conn.executescript(o.read())

Loading the lookup tables first. Using a for loop as there are 8 tables to populate.

In [11]:
load_lookup_tables = ( 
    'age_groups',
    'genders',
    'membership_levels',
    'location_zones',
    'item_categories',
    'brand_tiers',
    'device_types',
    'traffic_sources'
)

for table in load_lookup_tables:
    read_table = pd.read_csv(f'../data/clean/{table}.csv')
    read_table.to_sql(table, conn, if_exists='append', index=False)

Loading the interactions table last as it references all foreign keys from the looup tables.

In [12]:
main_table = pd.read_csv(f'../data/clean/interactions.csv')
main_table.to_sql('interactions', conn, if_exists='append', index=False)

12000

---
## 7. Verify

Row counts per table, and one join that returns the rows you expect. Then open `data/project.db` in DB Browser and look at it.

Checking the number of rows in each table is correct to ensure they were properly loaded.

In [ ]:
#row counts per table
#run_query(actual query here in """ """) function

list_of_tbls = [
    'interactions',
    'age_groups',
    'genders',
    'membership_levels',
    'location_zones',
    'item_categories',
    'brand_tiers',
    'device_types',
    'traffic_sources'
]

for tbl in list_of_tbls:
    query = f""" 
        SELECT COUNT(*) as row_count
        FROM {tbl}
    """
    result = run_query(query)
    print(f"{tbl}: {result.iloc[0]['row_count']}")


interactions: 12000
age_groups: 5
genders: 3
membership_levels: 4
location_zones: 4
item_categories: 10
brand_tiers: 4
device_types: 3
traffic_sources: 6


Running a JOIN query to ensure it returns the rows as expected.

In [11]:
query = f"""
    SELECT 
        DISTINCT b.brand_tier,
		COUNT(*) as count
    FROM interactions i 
    JOIN brand_tiers b
	    ON i.brand_tier_id = b.brand_tier_id
    GROUP BY b.brand_tier
"""
run_query(query)

,brand_tier,count
0,Budget,3706
1,Luxury,961
2,Premium,2582
3,Standard,4751
